# Per-label vs joint XGBoost under `MultioutputScorer`

Same wide **binary-multilabel** data, three model structures behind the *same*
`MultioutputScorer` API:

1. **Per-label** — `MultiOutputClassifierEstimator(XGBClassifier)`: **N independent** boosters.
2. **Joint, `one_output_per_tree`** — `JointXGBMultiOutputClassifierEstimator` (default): **one** booster, separate trees per label, **GPU-capable, no cross-label learning**.
3. **Joint, `multi_output_tree`** — vector-leaf shared splits: **one** booster with **genuine cross-label learning** (CPU-only).

Key point: *"joint" means one model artifact — it does **not** automatically mean
cross-label learning.* Only `multi_output_tree` (and sklearn tree ensembles) share
tree structure across labels. The scorer's output contract is identical in all cases.

In [1]:
# ruff: noqa: E402  (thread-pool env vars must be set before imports)
# macOS BLAS/OpenMP guard — set BEFORE numpy/xgboost import.
import os

for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

import skrec.constants as C
from skrec.estimator.classification.joint_xgb_multioutput import JointXGBMultiOutputClassifierEstimator
from skrec.estimator.classification.multioutput_classifier import MultiOutputClassifierEstimator
from skrec.scorer.multioutput import MultioutputScorer

## Synthetic wide multilabel data

Three **correlated** binary labels (so a cross-label model has something to exploit):
`ITEM_b` shares signal with `ITEM_a`. One row per user; features are plain columns.

In [2]:
def make_multilabel(n=4000, seed=42):
    rng = np.random.default_rng(seed)
    feats = pd.DataFrame(rng.random((n, 6)), columns=[f"f{i}" for i in range(6)])
    df = feats.copy()
    df[C.USER_ID_NAME] = np.arange(n).astype(str)
    df["ITEM_a"] = ((feats.f0 + rng.random(n) * 0.3) > 0.65).astype(int)
    df["ITEM_b"] = ((feats.f0 + feats.f1 + rng.random(n) * 0.3) > 1.05).astype(int)  # correlated with a
    df["ITEM_c"] = ((feats.f2 + rng.random(n) * 0.3) > 0.65).astype(int)
    return df[[C.USER_ID_NAME, "ITEM_a", "ITEM_b", "ITEM_c"] + [f"f{i}" for i in range(6)]]


LABELS = ["ITEM_a", "ITEM_b", "ITEM_c"]
df = make_multilabel()
train_df, test_df = df.iloc[:3200].copy(), df.iloc[3200:].copy()
print("train:", train_df.shape, "test:", test_df.shape)
print("label rates:\n", df[LABELS].mean())

train: (3200, 10) test: (800, 10)
label rates:
 ITEM_a    0.50250
ITEM_b    0.59575
ITEM_c    0.49775
dtype: float64


In [3]:
def evaluate(scorer):
    """Train on train_df, return per-label test ROC-AUC + wall-clock train time."""
    X, y = scorer.process_datasets(interactions_df=train_df)
    t0 = time.perf_counter()
    scorer.train_model(X, y)
    train_s = time.perf_counter() - t0
    proba = scorer.score_items_per_target(interactions=test_df)  # (n_test, N) P(label=1)
    aucs = {lab: roc_auc_score(test_df[lab].to_numpy(), proba[lab].to_numpy()) for lab in LABELS}
    return train_s, aucs


PARAMS = {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.1}

## 1. Per-label — N independent boosters

In [4]:
per_label_est = MultiOutputClassifierEstimator(XGBClassifier, dict(PARAMS, objective="binary:logistic"))
per_label_scorer = MultioutputScorer(per_label_est)
t_per_label, auc_per_label = evaluate(per_label_scorer)
n_boosters_per_label = len(per_label_est._model.estimators_)
print(f"# booster objects: {n_boosters_per_label}  (one per label)")
print(f"train time: {t_per_label:.2f}s")
print("per-label test ROC-AUC:", {k: round(v, 4) for k, v in auc_per_label.items()})

2026-06-12 00:54:11,340 - skrec.scorer.base_scorer - INFO Receiving DataFrames for Interactions and Users


2026-06-12 00:54:11,340 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (800, 10)


2026-06-12 00:54:11,340 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (800, 1)


2026-06-12 00:54:11,341 - skrec.scorer.base_scorer - INFO Merging DataFrames


2026-06-12 00:54:11,341 - skrec.scorer.base_scorer - INFO Completed Merging User-Interactions DataFrames


# booster objects: 3  (one per label)
train time: 0.11s
per-label test ROC-AUC: {'ITEM_a': 0.9804, 'ITEM_b': 0.9847, 'ITEM_c': 0.9788}


## 2. Joint, `one_output_per_tree` (default) — one booster, no cross-label learning

In [5]:
joint_indep_est = JointXGBMultiOutputClassifierEstimator(dict(PARAMS))  # default multi_strategy
joint_indep_scorer = MultioutputScorer(joint_indep_est)
t_joint_indep, auc_joint_indep = evaluate(joint_indep_scorer)
print("# booster objects: 1  (single joint XGBClassifier, multi_strategy='one_output_per_tree')")
print(f"train time: {t_joint_indep:.2f}s")
print("per-label test ROC-AUC:", {k: round(v, 4) for k, v in auc_joint_indep.items()})

2026-06-12 00:54:11,447 - skrec.scorer.base_scorer - INFO Receiving DataFrames for Interactions and Users


2026-06-12 00:54:11,447 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (800, 10)


2026-06-12 00:54:11,447 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (800, 1)


2026-06-12 00:54:11,447 - skrec.scorer.base_scorer - INFO Merging DataFrames


2026-06-12 00:54:11,448 - skrec.scorer.base_scorer - INFO Completed Merging User-Interactions DataFrames


# booster objects: 1  (single joint XGBClassifier, multi_strategy='one_output_per_tree')
train time: 0.10s
per-label test ROC-AUC: {'ITEM_a': 0.9804, 'ITEM_b': 0.9847, 'ITEM_c': 0.9788}


## 3. Joint, `multi_output_tree` — one booster, genuine cross-label learning (CPU-only)

Splits are chosen on the summed gradient across all labels, so labels share tree
structure. This is the only one of the three that does cross-label learning.

In [6]:
joint_cross_est = JointXGBMultiOutputClassifierEstimator(dict(PARAMS, multi_strategy="multi_output_tree"))
joint_cross_scorer = MultioutputScorer(joint_cross_est)
t_joint_cross, auc_joint_cross = evaluate(joint_cross_scorer)
print("# booster objects: 1  (single joint XGBClassifier, multi_strategy='multi_output_tree')")
print(f"train time: {t_joint_cross:.2f}s")
print("per-label test ROC-AUC:", {k: round(v, 4) for k, v in auc_joint_cross.items()})

2026-06-12 00:54:11,456 - skrec.estimator.classification.joint_xgb_multioutput - INFO JointXGBMultiOutputClassifierEstimator: multi_strategy='multi_output_tree' (vector leaf) is active — labels share tree structure, so this enables genuine cross-label learning. Note: vector-leaf trees are CPU-only in XGBoost.


2026-06-12 00:54:11,596 - skrec.scorer.base_scorer - INFO Receiving DataFrames for Interactions and Users


2026-06-12 00:54:11,596 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (800, 10)


2026-06-12 00:54:11,596 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (800, 1)


2026-06-12 00:54:11,596 - skrec.scorer.base_scorer - INFO Merging DataFrames


2026-06-12 00:54:11,597 - skrec.scorer.base_scorer - INFO Completed Merging User-Interactions DataFrames


# booster objects: 1  (single joint XGBClassifier, multi_strategy='multi_output_tree')
train time: 0.14s
per-label test ROC-AUC: {'ITEM_a': 0.9824, 'ITEM_b': 0.9848, 'ITEM_c': 0.9792}


## Side-by-side

In [7]:
summary = pd.DataFrame(
    {
        "per_label (N boosters)": {
            "# boosters": n_boosters_per_label,
            "train_s": round(t_per_label, 2),
            **auc_per_label,
        },
        "joint one_output_per_tree": {"# boosters": 1, "train_s": round(t_joint_indep, 2), **auc_joint_indep},
        "joint multi_output_tree": {"# boosters": 1, "train_s": round(t_joint_cross, 2), **auc_joint_cross},
    }
).T
summary[LABELS] = summary[LABELS].astype(float).round(4)
print("Same MultioutputScorer API + identical output shape for all three:")
print("  score_items cols:", list(per_label_scorer.score_items(interactions=test_df.head(2)).columns))
summary

2026-06-12 00:54:11,605 - skrec.scorer.base_scorer - INFO Receiving DataFrames for Interactions and Users


2026-06-12 00:54:11,605 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (2, 10)


2026-06-12 00:54:11,605 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (2, 1)


2026-06-12 00:54:11,606 - skrec.scorer.base_scorer - INFO Merging DataFrames


2026-06-12 00:54:11,606 - skrec.scorer.base_scorer - INFO Completed Merging User-Interactions DataFrames


Same MultioutputScorer API + identical output shape for all three:
  score_items cols: ['ITEM_a_0', 'ITEM_a_1', 'ITEM_b_0', 'ITEM_b_1', 'ITEM_c_0', 'ITEM_c_1']


,# boosters,train_s,ITEM_a,ITEM_b,ITEM_c
per_label (N boosters),3.0,0.11,0.9804,0.9847,0.9788
joint one_output_per_tree,1.0,0.10,0.9804,0.9847,0.9788
joint multi_output_tree,1.0,0.14,0.9824,0.9848,0.9792


**Reading the table.** All three share one `MultioutputScorer` API and identical
output shapes. Per-label and joint-`one_output_per_tree` are close on metrics
(neither shares cross-label signal); `multi_output_tree` is the one that can move
correlated-label AUC by sharing structure — at the cost of being CPU-only. Pick by
constraint: GPU + many trees → per-label or `one_output_per_tree`; correlated labels
where shared structure helps and CPU is acceptable → `multi_output_tree`.

## Class weighting (works on any of them)

In [8]:
# 'balanced' up-weights rare positives at fit time — no special API, just a constructor arg.
imbalanced = make_multilabel(seed=7)
imbalanced["ITEM_c"] = (imbalanced.f2 > 0.9).astype(int)  # make c rare (~10%)
itr, ite = imbalanced.iloc[:3200], imbalanced.iloc[3200:]


def mean_pos_c(sample_weight):
    sc = MultioutputScorer(JointXGBMultiOutputClassifierEstimator(dict(PARAMS), sample_weight=sample_weight))
    X, y = sc.process_datasets(interactions_df=itr)
    sc.train_model(X, y)
    return float(sc.score_items_per_target(interactions=ite)["ITEM_c"].mean())


print("rare-label ITEM_c positive rate:", round(imbalanced["ITEM_c"].mean(), 3))
print("mean P(ITEM_c=1)  unweighted:", round(mean_pos_c(None), 4))
print("mean P(ITEM_c=1)  balanced:  ", round(mean_pos_c("balanced"), 4), " (higher — rare class up-weighted)")

rare-label ITEM_c positive rate: 0.106


2026-06-12 00:54:11,695 - skrec.scorer.base_scorer - INFO Receiving DataFrames for Interactions and Users


2026-06-12 00:54:11,696 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (800, 10)


2026-06-12 00:54:11,696 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (800, 1)


2026-06-12 00:54:11,696 - skrec.scorer.base_scorer - INFO Merging DataFrames


...

2026-06-12 00:54:11,778 - skrec.scorer.base_scorer - INFO Shape of Interactions DataFrame: (800, 10)


2026-06-12 00:54:11,779 - skrec.scorer.base_scorer - INFO Shape of Users DataFrame: (800, 1)


2026-06-12 00:54:11,779 - skrec.scorer.base_scorer - INFO Merging DataFrames


2026-06-12 00:54:11,779 - skrec.scorer.base_scorer - INFO Completed Merging User-Interactions DataFrames


mean P(ITEM_c=1)  unweighted: 0.119
mean P(ITEM_c=1)  balanced:   0.1214  (higher — rare class up-weighted)
